# 07 Geologic Hazards and Evidence Boundaries

Hazard evidence is separated into geology-supported, public-soils-supported, field-supported, and unknown. This notebook does not derive a reservation-wide soil-hazard map from absent Pine Ridge SSURGO coverage.

In [1]:
# Imports
import sys
from pathlib import Path
REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/"src").is_dir())
sys.path.insert(0, str(REPO_ROOT)) if str(REPO_ROOT) not in sys.path else None
import pandas as pd
import geopandas as gpd
import yaml
from IPython.display import display
from src.constants import REPO_ROOT as ROOT, OUTPUTS_DIR
from src.loaders import load_tribal_boundaries
from src.sovereignty import print_data_acknowledgment, generate_citations
with open(ROOT/"config"/"config.yaml") as stream: CONFIG = yaml.safe_load(stream)
primary = load_tribal_boundaries(["Pine Ridge"])
pine_ridge = primary[primary["NAME"] == "Pine Ridge"]

In [2]:
# Print data acknowledgement at the top of every notebook
print_data_acknowledgment(["usda_ssurgo", "usgs_3d_model"])

TRIBAL SOILS AND GEOLOGY DATA GOVERNANCE ACKNOWLEDGMENT

This analysis uses data that describes the lands and subsurface
resources of the Oglala Sioux Tribe and the Oglala Lakota people.
peoples. This data is governed by the following frameworks:

OCAP®  : Tribal Nations have the right to Ownership, Control,
         Access, and Possession of data about their lands,
         including subsurface geological and soil data.
         Reference: https://fnigc.ca/ocap-training/

CARE   : Data use must deliver Collective Benefit to Indigenous
         peoples, respect their Authority to Control, uphold
         Responsibility to communities, and center Ethics.
         Reference: https://www.gida-global.org/care

FAIR   : Data is Findable, Accessible, Interoperable, Reusable.
         FAIR governs technical standards; CARE and OCAP® govern
         the ethical obligations FAIR alone does not address.
         Reference: https://www.go-fair.org/fair-principles/

IEEE 2890-2025 : Recommended Pr

In [3]:
from src.soil_evidence import evidence_register
register = evidence_register()
display(register)

,source,valid_scope,evidence_level,constraint
0,Modeled geology,regional,geology-supported,Formation tops; not soil properties
1,Public SSURGO,verified polygons only,public-soils-supported,No spatial extrapolation
2,Tribal field profiles,authorized records only,field-supported,Governance gate required
3,Pine Ridge soils without authorized observations,unknown,unknown,No soil inference


## Geology-supported evidence

The Spangler model can support statements about modeled formation-top elevation and relative subsurface position. It cannot by itself supply soil texture, shrink–swell measurements, erodibility, or site-specific geotechnical properties.

In [4]:
depth_figure = ROOT/"outputs"/"figures"/"04_depth_to_pierre_shale.png"
cross_section = ROOT/"outputs"/"04_pine_ridge_cross_section.csv"
display(pd.DataFrame([
    ["Depth-to-Pierre diagnostic", depth_figure.exists(), "geology-supported", "DEM/model datum and resolution uncertainty applies"],
    ["Modeled horizon cross-section", cross_section.exists(), "geology-supported", "regional model; not site investigation"],
], columns=["artifact", "available", "evidence_level", "constraint"]))

,artifact,available,evidence_level,constraint
0,Depth-to-Pierre diagnostic,True,geology-supported,DEM/model datum and resolution uncertainty app...
1,Modeled horizon cross-section,True,geology-supported,regional model; not site investigation


## Public-soils coverage gate

Expansive-soil, hydrologic-group, farmland, and erodibility outputs require verified local soil polygons and attributes. The gate prevents adjacent surveys from becoming implicit Pine Ridge estimates.

In [5]:
from src.loaders import load_ssurgo_mapunits
from src.soil_evidence import require_coverage, SoilCoverageError
mapunits = load_ssurgo_mapunits()
try:
    require_coverage(mapunits, pine_ridge, "Pine Ridge public SSURGO")
    public_soil_hazards_allowed = True
except SoilCoverageError as exc:
    public_soil_hazards_allowed = False
    print(exc)
if not public_soil_hazards_allowed:
    print("No reservation-wide SSURGO soil-hazard classification is produced.")

Pine Ridge public SSURGO covers 0.0% of the requested geography; 95.0% is required. Analysis stopped—adjacent data are context only.
No reservation-wide SSURGO soil-hazard classification is produced.


C:\Users\gekek\AppData\Local\Temp\ipykernel_32416\2801918890.py:3: UserWarning: No SSURGO GDB found in data/raw/ssurgo/. Download from https://websoilsurvey.nrcs.usda.gov/ using the ESRI Soil Data Downloader. See docs/data_intake_guide.md.
  mapunits = load_ssurgo_mapunits()


## Field-supported evidence

Authorized field observations can support sampled-site interpretations after governance and quality gates. Unsampled Pine Ridge soils remain `unknown`; absence of data is not evidence of low hazard or uniform conditions.

In [6]:
hazard_summary = pd.DataFrame([
    ["Pierre Shale proximity", "geology-supported", "Screening only; verify datum and investigate site"],
    ["Expansive soil", "unknown without authorized soil measurements", "Do not map reservation-wide"],
    ["Soil erodibility", "unknown without authorized soil measurements", "Do not infer from adjacent counties"],
    ["Site geotechnical suitability", "site investigation required", "No regional dataset substitutes for design investigation"],
], columns=["question", "current_evidence", "permitted_interpretation"])
display(hazard_summary)

,question,current_evidence,permitted_interpretation
0,Pierre Shale proximity,geology-supported,Screening only; verify datum and investigate site
1,Expansive soil,unknown without authorized soil measurements,Do not map reservation-wide
2,Soil erodibility,unknown without authorized soil measurements,Do not infer from adjacent counties
3,Site geotechnical suitability,site investigation required,No regional dataset substitutes for design inv...
